In [1]:
!pip install Sastrawi requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.6 MB/s eta 0:00:00


In [ ]:
import requests
import json
import re
import pandas as pd
import time
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# --- KONFIGURASI MULTI-KEYWORD ---
API_KEY = "GANTI API KEY ANDA"
DAFTAR_KATA_KUNCI = ["kemiskinan", "ketimpangan", "kesejahteraan", "pengeluaran"]

factory_stop = StopWordRemoverFactory()
stopword = factory_stop.create_stop_word_remover()

dokumen_mentah = []

print("Mulai mengunduh data dari WebAPI BPS (Multi-Keyword)...")
for keyword in DAFTAR_KATA_KUNCI:
    print(f"Mencari keyword: '{keyword}'...")
    for page in range(1, 6): # Ambil 5 halaman per keyword
        url = f"https://webapi.bps.go.id/v1/api/list/model/pressrelease/domain/0000/keyword/{keyword}/page/{page}/key/{API_KEY}/"

        try:
            response = requests.get(url)
            time.sleep(1)
            if response.status_code == 200:
                data = response.json()
                if data.get('data') and isinstance(data['data'], list) and len(data['data']) > 1 and data['data'][1]:
                    for pub in data['data'][1]:
                        judul = pub.get('title', '')
                        abstrak = pub.get('abstract', '')
                        if abstrak:
                            dokumen_mentah.append({
                                "id": pub.get('brs_id'),
                                "tahun": str(pub.get('rl_date', ''))[:4],
                                "judul": judul,
                                "teks": f"{judul}. {abstrak}"
                            })
        except Exception as e:
            continue

# Hapus duplikat berdasarkan ID BRS (karena bisa saja satu BRS muncul di keyword berbeda)
df_tmp = pd.DataFrame(dokumen_mentah).drop_duplicates(subset=['id'])
print(f"\nTotal dokumen unik terkumpul: {len(df_tmp)} dokumen.")

# --- PRA-PEMROSESAN & CHUNKING (MEMPECAH PARAGRAF AGAR JUMLAH AMAN) ---
dokumen_bersih = []

for idx, row in df_tmp.iterrows():
    teks_full = row['teks'].lower()
    # Bersihkan simbol
    teks_full = re.sub(r'[^a-z\s]', ' ', teks_full)
    teks_clean = stopword.remove(teks_full)
    tokens = teks_clean.split()

    dokumen_bersih.append({
        "tahun": row["tahun"],
        "judul": row["judul"],
        "tokens": tokens,
        "teks_bersih": teks_clean
    })

# Simpan ke JSON
with open('data_teks_kemiskinan.json', 'w', encoding='utf-8') as f:
    json.dump(dokumen_bersih, f, ensure_ascii=False, indent=4)

print(f"Berhasil! File JSON siap dengan total {len(dokumen_bersih)} dokumen.")

Mulai mengunduh data dari WebAPI BPS (Multi-Keyword)...
Mencari keyword: 'kemiskinan'...
Mencari keyword: 'ketimpangan'...
Mencari keyword: 'kesejahteraan'...
Mencari keyword: 'pengeluaran'...

Total dokumen unik terkumpul: 114 dokumen.
Berhasil! File JSON siap dengan total 114 dokumen.
